# Distributional Regression with NAMpy

This notebook demonstrates how to use NAMpy's Location-Scale-Shape (LSS) models for distributional regression. Unlike standard regression that only predicts the mean, distributional regression models the full conditional distribution of the target variable.

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

# Import NAMpy LSS models
from nampy.models import NAMLSS, NBMLSS

np.random.seed(42)

## 1. Understanding Distributional Regression

In distributional regression, we model the parameters of a probability distribution (e.g., mean and variance for Normal distribution) as functions of input features. This allows us to:

- Capture **heteroscedasticity** (varying uncertainty)
- Provide **prediction intervals**
- Model **asymmetric distributions**

Let's create a synthetic dataset with heteroscedastic noise to demonstrate.

In [ ]:
# Generate heteroscedastic data
n_samples = 2000

# Features
X = np.random.uniform(-3, 3, (n_samples, 3))

# True mean function
true_mean = 2 * X[:, 0] + np.sin(2 * X[:, 1]) + 0.5 * X[:, 2]**2

# True variance function (heteroscedastic - variance depends on features)
true_std = 0.5 + 0.3 * np.abs(X[:, 0]) + 0.2 * np.abs(X[:, 1])

# Generate target with heteroscedastic noise
y = true_mean + true_std * np.random.randn(n_samples)

X_df = pd.DataFrame(X, columns=['x1', 'x2', 'x3'])

print(f"Data shape: {X_df.shape}")
print(f"Target range: [{y.min():.2f}, {y.max():.2f}]")

In [ ]:
# Visualize the heteroscedastic nature
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for i, col in enumerate(['x1', 'x2', 'x3']):
    axes[i].scatter(X_df[col], y, alpha=0.3, s=10)
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('y')
    axes[i].set_title(f'y vs {col}')

plt.suptitle('Heteroscedastic Data: Notice varying spread across feature values', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X_df, y, test_size=0.2, random_state=42
)

# Also keep the true parameters for evaluation
_, true_mean_test, _, true_std_test = train_test_split(
    X_df, true_mean, true_std, test_size=0.2, random_state=42
)[1::2]

print(f"Training samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")

## 2. Training a NAMLSS Model

NAMLSS models the parameters of a distribution. For the Normal distribution, it predicts both the mean (location) and standard deviation (scale).

In [ ]:
# Train NAMLSS with Normal distribution
model = NAMLSS(
    numerical_preprocessing="standardization",
    dropout=0.1,
    layer_sizes=[64, 32, 16],
)

model.fit(
    X_train, 
    y_train, 
    max_epochs=150,
    lr=1e-3,
    patience=15,
    batch_size=128,
    family="normal"  # Use Normal distribution
)

In [ ]:
# Get distribution parameters
params = model.predict(X_test)

print(f"Predicted parameters shape: {params.shape}")
print(f"Parameters: [mean, std] for each sample")

pred_mean = params[:, 0]
pred_std = params[:, 1]

In [ ]:
# Evaluate mean predictions
mse = mean_squared_error(y_test, pred_mean)
r2 = r2_score(y_test, pred_mean)

print("Mean Prediction Performance:")
print(f"  MSE: {mse:.4f}")
print(f"  RMSE: {np.sqrt(mse):.4f}")
print(f"  R²: {r2:.4f}")

In [ ]:
# Evaluate uncertainty estimation
std_correlation = np.corrcoef(true_std_test, pred_std)[0, 1]
print(f"\nUncertainty Estimation:")
print(f"  Correlation between true and predicted std: {std_correlation:.4f}")

## 3. Visualizing Prediction Intervals

One key benefit of distributional regression is the ability to provide prediction intervals.

In [ ]:
# Sort by x1 for visualization
sort_idx = np.argsort(X_test['x1'].values)
x1_sorted = X_test['x1'].values[sort_idx]
y_sorted = y_test[sort_idx]
mean_sorted = pred_mean[sort_idx]
std_sorted = pred_std[sort_idx]

# Create prediction intervals (95% CI)
lower_95 = mean_sorted - 1.96 * std_sorted
upper_95 = mean_sorted + 1.96 * std_sorted

# 68% CI (1 std)
lower_68 = mean_sorted - std_sorted
upper_68 = mean_sorted + std_sorted

plt.figure(figsize=(12, 6))
plt.fill_between(x1_sorted, lower_95, upper_95, alpha=0.2, color='blue', label='95% CI')
plt.fill_between(x1_sorted, lower_68, upper_68, alpha=0.3, color='blue', label='68% CI')
plt.scatter(x1_sorted, y_sorted, alpha=0.4, s=10, c='gray', label='Actual')
plt.plot(x1_sorted, mean_sorted, 'b-', lw=2, label='Predicted Mean')
plt.xlabel('x1')
plt.ylabel('y')
plt.title('NAMLSS Prediction Intervals')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Check calibration of prediction intervals
in_95_ci = np.mean((y_test >= lower_95[np.argsort(sort_idx)]) & 
                   (y_test <= upper_95[np.argsort(sort_idx)]))
in_68_ci = np.mean((y_test >= lower_68[np.argsort(sort_idx)]) & 
                   (y_test <= upper_68[np.argsort(sort_idx)]))

print("Prediction Interval Calibration:")
print(f"  95% CI coverage: {in_95_ci:.2%} (expected: 95%)")
print(f"  68% CI coverage: {in_68_ci:.2%} (expected: 68%)")

## 4. Comparing Predicted vs True Variance

Let's see how well the model captures the heteroscedasticity.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# True vs predicted mean
axes[0].scatter(true_mean_test, pred_mean, alpha=0.5, s=20)
axes[0].plot([true_mean_test.min(), true_mean_test.max()], 
             [true_mean_test.min(), true_mean_test.max()], 'r--', lw=2)
axes[0].set_xlabel('True Mean')
axes[0].set_ylabel('Predicted Mean')
axes[0].set_title(f'Mean: Correlation = {np.corrcoef(true_mean_test, pred_mean)[0,1]:.4f}')

# True vs predicted std
axes[1].scatter(true_std_test, pred_std, alpha=0.5, s=20, c='orange')
axes[1].plot([true_std_test.min(), true_std_test.max()], 
             [true_std_test.min(), true_std_test.max()], 'r--', lw=2)
axes[1].set_xlabel('True Std')
axes[1].set_ylabel('Predicted Std')
axes[1].set_title(f'Std: Correlation = {np.corrcoef(true_std_test, pred_std)[0,1]:.4f}')

plt.tight_layout()
plt.show()

## 5. Different Distribution Families

NAMpy supports multiple distribution families. Let's explore using different distributions.

In [ ]:
# Generate data with positive, right-skewed target (good for Gamma distribution)
X_gamma = np.random.uniform(0, 3, (1500, 3))
shape_param = 2 + X_gamma[:, 0]  # Shape parameter depends on feature
scale_param = 1 + 0.5 * X_gamma[:, 1]  # Scale parameter depends on feature
y_gamma = np.random.gamma(shape_param, scale_param)

X_gamma_df = pd.DataFrame(X_gamma, columns=['x1', 'x2', 'x3'])

# Visualize the skewed distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(y_gamma, bins=50, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('y')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Target Distribution (Right-skewed)')

axes[1].scatter(X_gamma_df['x1'], y_gamma, alpha=0.3, s=10)
axes[1].set_xlabel('x1')
axes[1].set_ylabel('y')
axes[1].set_title('y vs x1')

plt.tight_layout()
plt.show()

In [ ]:
# Split data
X_train_g, X_test_g, y_train_g, y_test_g = train_test_split(
    X_gamma_df, y_gamma, test_size=0.2, random_state=42
)

# Train with Gamma family
model_gamma = NAMLSS(
    numerical_preprocessing="standardization",
    dropout=0.1,
    layer_sizes=[64, 32],
)

model_gamma.fit(
    X_train_g, 
    y_train_g, 
    max_epochs=100,
    lr=1e-3,
    patience=10,
    batch_size=128,
    family="gamma"  # Use Gamma distribution
)

In [ ]:
# Get Gamma distribution parameters
params_gamma = model_gamma.predict(X_test_g)
print(f"Gamma distribution parameters shape: {params_gamma.shape}")

# For Gamma distribution, params are typically [concentration, rate]
# Mean = concentration / rate
pred_mean_gamma = params_gamma[:, 0] / params_gamma[:, 1]

# Evaluate
r2_gamma = r2_score(y_test_g, pred_mean_gamma)
print(f"\nGamma model R²: {r2_gamma:.4f}")

## 6. Probabilistic Metrics

For distributional regression, we can evaluate using probabilistic metrics like the Continuous Ranked Probability Score (CRPS).

In [ ]:
# Calculate log-likelihood for Normal model
def normal_log_likelihood(y, mean, std):
    return -0.5 * np.log(2 * np.pi * std**2) - 0.5 * ((y - mean) / std)**2

log_lik = normal_log_likelihood(y_test, pred_mean, pred_std)
mean_log_lik = np.mean(log_lik)

print(f"Mean Log-Likelihood: {mean_log_lik:.4f}")

In [ ]:
# Compare with a constant variance model
# (simulating what a standard regression would give)
constant_std = np.std(y_train - np.mean(y_train))
log_lik_constant = normal_log_likelihood(y_test, pred_mean, np.full_like(pred_std, constant_std))
mean_log_lik_constant = np.mean(log_lik_constant)

print(f"Comparison:")
print(f"  NAMLSS Log-Likelihood: {mean_log_lik:.4f}")
print(f"  Constant Variance Log-Likelihood: {mean_log_lik_constant:.4f}")
print(f"  Improvement: {mean_log_lik - mean_log_lik_constant:.4f}")

## 7. Shape Functions for Distribution Parameters

We can interpret how features affect both the location (mean) and scale (variance) parameters.

In [ ]:
# Get shape function outputs
shape_outputs = model.get_shape_function_outputs(X_test)

# The outputs are for both mean and std parameters
n_features = X_test.shape[1]
shape_mean = shape_outputs[:, :n_features]
shape_std = shape_outputs[:, n_features:]

# Plot shape functions for mean parameter
fig, axes = plt.subplots(2, 3, figsize=(14, 8))

for i, col in enumerate(X_test.columns):
    # Mean shape function
    ax = axes[0, i]
    sort_idx = np.argsort(X_test[col].values)
    ax.scatter(X_test[col].values[sort_idx], shape_mean[:, i][sort_idx], alpha=0.3, s=10)
    ax.set_xlabel(col)
    ax.set_ylabel('Contribution to Mean')
    ax.set_title(f'Mean Shape: {col}')
    ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
    
    # Std shape function
    ax = axes[1, i]
    ax.scatter(X_test[col].values[sort_idx], shape_std[:, i][sort_idx], alpha=0.3, s=10, c='orange')
    ax.set_xlabel(col)
    ax.set_ylabel('Contribution to Std')
    ax.set_title(f'Std Shape: {col}')
    ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)

plt.suptitle('Shape Functions for Distribution Parameters', fontsize=14)
plt.tight_layout()
plt.show()

## Summary

In this notebook, we learned how to:

1. **Use NAMLSS for distributional regression** to model full distributions
2. **Predict distribution parameters** (mean and variance for Normal distribution)
3. **Create prediction intervals** for uncertainty quantification
4. **Evaluate probabilistic predictions** using log-likelihood
5. **Use different distribution families** (Normal, Gamma, etc.)
6. **Interpret how features affect both location and scale** through shape functions

Distributional regression is particularly useful when:
- The data is heteroscedastic (varying uncertainty)
- You need prediction intervals
- The target follows a non-Normal distribution
- Understanding uncertainty is as important as point predictions